## 模型评估与验证代码模板

---

### 模板一：基础版（训练/测试二拆分）

适用场景：数据量大，不需要反复调参。

```python
from sklearn.model_selection import train_test_split
from sklearn.xxx import SomeModel

# 1. 加载数据
X, y = load_data()

# 2. 拆分
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 训练
model = SomeModel()
model.fit(Xtr, ytr)

# 4. 评估
print("训练集:", model.score(Xtr, ytr))
print("测试集:", model.score(Xte, yte))
```

---

### 模板二：标准版（训练/验证/测试三拆分）

适用场景：需要手动调参，数据量中等。

```python
# 1. 加载数据
X, y = load_data()

# 2. 第一次拆分：隔离测试集
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 第二次拆分：分出验证集
Xtr_tr, Xva, ytr_tr, yva = train_test_split(Xtr, ytr, test_size=0.2, random_state=42)

# 4. 在验证集上调参
best_score = -1
best_param = None

for param in candidate_params:
    model = SomeModel(param=param)
    model.fit(Xtr_tr, ytr_tr)
    score = model.score(Xva, yva)
    
    if score > best_score:
        best_score = score
        best_param = param

# 5. 用最优参数在测试集上最终评估
final_model = SomeModel(param=best_param)
final_model.fit(Xtr, ytr)                    # ← 注意：在全量训练集上重训
print("测试集:", final_model.score(Xte, yte))
```

---

### 模板三：Pipeline版（防数据泄露）

适用场景：有预处理步骤（缩放、编码等），需要保证一致。

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1-3. 数据拆分同上...

# 4. Pipeline 调参
best_score = -1
best_C = None

for C in [0.01, 0.1, 1, 10]:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SomeModel(C=C))
    ])
    pipe.fit(Xtr_tr, ytr_tr)
    score = pipe.score(Xva, yva)
    
    if score > best_score:
        best_score = score
        best_C = C

# 5. 最终评估
final_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SomeModel(C=best_C))
])
final_pipe.fit(Xtr, ytr)                     # 全量训练集
print("测试集:", final_pipe.score(Xte, yte))
```

---

### 模板四：交叉验证版（数据量少时）

适用场景：数据量小，单次拆分不稳定。

```python
from sklearn.model_selection import cross_val_score, train_test_split

# 1-2. 先隔离测试集
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 用交叉验证调参
best_score = -1
best_param = None

for param in candidate_params:
    model = SomeModel(param=param)
    scores = cross_val_score(model, Xtr, ytr, cv=5)  # 5折交叉验证
    avg = scores.mean()
    
    if avg > best_score:
        best_score = avg
        best_param = param

# 4. 最终评估
final_model = SomeModel(param=best_param)
final_model.fit(Xtr, ytr)
print("测试集:", final_model.score(Xte, yte))
```

---

### 模板五：GridSearchCV 自动化版（最常用）

```python
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

# 1-2. 隔离测试集
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 构建 Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SomeModel())
])

# 4. 定义参数网格
param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__gamma": [0.01, 0.1, 1]
}

# 5. 网格搜索（内部自动做交叉验证）
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(Xtr, ytr)

# 6. 查看结果
print("最优参数:", grid.best_params_)
print("交叉验证得分:", grid.best_score_)
print("测试集得分:", grid.score(Xte, yte))
```

---

### 三大原则总结

```
┌──────────────┬──────────────────────────────────┐
│   数据角色    │            只能用于              │
├──────────────┼──────────────────────────────────┤
│   训练集      │  fit() 训练模型                  │
│   验证集      │  调参、选模型（可以反复看）        │
│   测试集      │  最终评估（只能用一次）            │
└──────────────┴──────────────────────────────────┘
```

| 原则 | 说明 |
|---|---|
| **测试集隔离** | 测试集从一开始就锁住，调参阶段绝不碰它 |
| **验证集选参** | 在验证集上反复试参数，挑出最优组合 |
| **重训再测试** | 找到最优参数后，在完整训练集上重新训练，再去测试集评估 |

---

### 选择建议

| 条件 | 用哪个模板 |
|---|---|
| 数据 > 10万，参数少 | 模板一 |
| 数据 1万~10万，手动调参 | 模板二/三 |
| 数据 < 1万 | 模板四 |
| 日常开发，需要自动化 | 模板五 |